# Model Quantization and Deployment Optimization

## Introduction

Notebook 06 trained a MobileNetV2 transfer-learning model on the cached Mel Spectrogram tensors generated in Notebook 03.

This notebook focuses on deployment optimization. It does not recreate dataset splits, regenerate features, or retrain previous models. Instead, it loads the saved MobileNetV2 artifacts and compares multiple post-training quantization strategies.

The first full static INT8 experiment caused a severe macro F1 drop. This notebook keeps that negative result visible and adds safer alternatives:

- full static INT8 quantization of the MobileNetV2 core,
- dynamic quantization of supported `Linear` layers,
- classifier-only dynamic quantization while keeping the convolutional feature extractor in FP32.

The comparison reports model size, CPU inference time, accuracy, macro F1, and weighted F1.

## Hypothesis

**H7 - Quantization Efficiency:** A quantized MobileNetV2 variant will reduce deployment cost while preserving test macro F1 close to the original FP32 model.

Because the dataset is imbalanced, macro F1 remains the most important metric. A quantized model is not considered successful if it improves size or latency but destroys minority-class performance.

For this notebook, a macro F1 drop larger than `0.05` is treated as unacceptable.

## Load Configuration

All existing dataset, model, and output locations are loaded from `src/utils/config.py`.

Notebook 07 reads quantization artifact paths from `src/utils/config.py`, matching the project pattern used in earlier notebooks.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_ROOT = Path('..').resolve()

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

import pandas as pd
import torch

from src.data.feature_extraction import load_mel_dataset
from src.models.mobilenet_model import create_mobilenet_model
from src.optimization.quantization import (
    build_classifier_dynamic_model,
    build_dynamic_linear_model,
    build_static_int8_model,
    measure_inference_speed,
    prepare_mobilenet_inner_inputs,
    save_deployment_model,
    select_quantization_backend,
    size_in_mb,
)
from src.training.evaluate_torch import evaluate_torch_model
from src.training.train import load_model
from src.training.train_torch import load_torch_model
from src.utils.config import (
    PROJECT_ROOT,
    BEST_MOBILENET_MODEL_PATH,
    CLASSIFIER_DYNAMIC_MOBILENET_MODEL_PATH,
    DYNAMIC_LINEAR_MOBILENET_MODEL_PATH,
    FP32_MOBILENET_DEPLOYMENT_PATH,
    MEL_TEST_LABELS_PATH,
    MEL_TEST_PATH,
    MEL_VALIDATION_LABELS_PATH,
    MEL_VALIDATION_PATH,
    MOBILENET_BATCH_SIZE,
    MOBILENET_LABEL_ENCODER_PATH,
    MOBILENET_LEARNING_RATE,
    MOBILENET_METRICS_SUMMARY_PATH,
    MOBILENET_OUTPUT_DIR,
    QUANTIZATION_CLASSIFICATION_REPORTS_PATH,
    QUANTIZATION_METRICS_SUMMARY_PATH,
    QUANTIZATION_SIZE_COMPARISON_PATH,
    QUANTIZATION_SPEED_COMPARISON_PATH,
    STATIC_INT8_MOBILENET_MODEL_PATH,
)

EVALUATION_DEVICE = 'cpu'
STATIC_CALIBRATION_SAMPLE_COUNT = 1024
TIMING_SAMPLE_COUNT = 256
TIMING_REPEATS = 5
ACCEPTABLE_MACRO_F1_DROP = 0.05

print('Project root:', PROJECT_ROOT)
print('Saved MobileNetV2 model:', BEST_MOBILENET_MODEL_PATH)
print('Saved MobileNetV2 label encoder:', MOBILENET_LABEL_ENCODER_PATH)
print('Evaluation device:', EVALUATION_DEVICE)
print('CUDA available:', torch.cuda.is_available())

Project root: C:\Softuni ML course\Project\music-genre-classification
Saved MobileNetV2 model: C:\Softuni ML course\Project\music-genre-classification\models\mobilenet\best_mobilenet_model.pt
Saved MobileNetV2 label encoder: C:\Softuni ML course\Project\music-genre-classification\models\mobilenet\mobilenet_label_encoder.joblib
Evaluation device: cpu
CUDA available: True


In [2]:
X_validation, labels_validation = load_mel_dataset(
    MEL_VALIDATION_PATH,
    MEL_VALIDATION_LABELS_PATH,
)

X_test, labels_test = load_mel_dataset(
    MEL_TEST_PATH,
    MEL_TEST_LABELS_PATH,
)

label_encoder = load_model(
    MOBILENET_LABEL_ENCODER_PATH
)

y_test = label_encoder.transform(
    labels_test
)

class_names = list(
    label_encoder.classes_
)

print('Validation tensors:', X_validation.shape)
print('Test tensors:', X_test.shape)
print('Classes:', class_names)

pd.Series(labels_test).value_counts().loc[class_names]

Validation tensors: (1415, 1, 128, 1206)
Test tensors: (1414, 1, 128, 1206)
Classes: [np.str_('blues'), np.str_('classical'), np.str_('country'), np.str_('hiphop'), np.str_('jazz'), np.str_('pop'), np.str_('rock')]


blues         26
classical    108
country       42
hiphop       193
jazz          72
pop           43
rock         930
Name: count, dtype: int64

## Load Saved MobileNetV2 Model

The MobileNetV2 architecture is recreated through the existing factory in `src/models/mobilenet_model.py`, then the saved Notebook 06 weights are loaded from the configured path.

No training is performed in this notebook.

In [3]:
mobilenet_fp32 = create_mobilenet_model(
    input_shape=X_test.shape[1:],
    num_classes=len(class_names),
    learning_rate=MOBILENET_LEARNING_RATE,
    train_base=False,
)

mobilenet_fp32 = load_torch_model(
    mobilenet_fp32,
    BEST_MOBILENET_MODEL_PATH,
    device=EVALUATION_DEVICE,
)

mobilenet_fp32.eval()

trainable_parameters = sum(
    parameter.numel()
    for parameter in mobilenet_fp32.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in mobilenet_fp32.parameters()
)

print('Trainable parameters:', trainable_parameters)
print('Total parameters:', total_parameters)
mobilenet_fp32

Trainable parameters: 8967
Total parameters: 2232839


MelMobileNetV2(
  (model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            

## Baseline Evaluation

The original FP32 MobileNetV2 model is evaluated on the cached test Mel Spectrogram tensors.

Evaluation is performed on CPU because all deployment comparisons target CPU inference.

In [4]:
if MOBILENET_METRICS_SUMMARY_PATH.exists():
    stored_mobilenet_metrics = pd.read_csv(
        MOBILENET_METRICS_SUMMARY_PATH
    )

    display(stored_mobilenet_metrics)
else:
    print('Stored MobileNetV2 metrics were not found.')

,model,split,accuracy,macro_f1,weighted_f1
0,mobilenet_v2,validation,0.828269,0.444825,0.789118
1,mobilenet_v2,test,0.813296,0.435781,0.770928


In [5]:
baseline_result = evaluate_torch_model(
    mobilenet_fp32,
    X_test,
    y_test,
    class_names,
    model_name='mobilenet_v2_fp32',
    split='test',
    batch_size=MOBILENET_BATCH_SIZE,
    device=EVALUATION_DEVICE,
)

baseline_summary = pd.DataFrame([
    baseline_result['summary']
])

baseline_summary

,model,split,accuracy,macro_f1,weighted_f1
0,mobilenet_v2_fp32,test,0.813296,0.435781,0.770928


In [6]:
pd.DataFrame(
    baseline_result['classification_report']
).transpose()

,precision,recall,f1-score,support
blues,1.000000,0.153846,0.266667,26.000000
classical,0.796296,0.796296,0.796296,108.000000
country,0.000000,0.000000,0.000000,42.000000
hiphop,0.804348,0.766839,0.785146,193.000000
jazz,0.750000,0.166667,0.272727,72.000000
pop,0.333333,0.023256,0.043478,43.000000
rock,0.818016,0.966667,0.886151,930.000000
accuracy,0.813296,0.813296,0.813296,0.813296
macro avg,0.643142,0.410510,0.435781,1414.000000
weighted avg,0.775338,0.813296,0.770928,1414.000000


## Quantization

Three quantization strategies are compared.

1. **Static INT8 MobileNetV2 core**: FX graph mode static quantization is applied to the inner MobileNetV2 network. Mel preprocessing and ImageNet normalization stay in FP32. Calibration uses validation data only, with a larger calibration sample count than the first experiment.

2. **Dynamic Linear-only quantization**: PyTorch dynamically quantizes supported `Linear` layers. Since MobileNetV2 is mostly convolutional, this is a safer but smaller optimization.

3. **Classifier-only dynamic quantization**: Only the classifier head is dynamically quantized. The convolutional feature extractor remains FP32.

The static INT8 result is intentionally retained even if it performs poorly, because a negative deployment result is still an important experiment outcome.

In [7]:
quantization_backend = select_quantization_backend()
torch.backends.quantized.engine = quantization_backend

print('Supported quantization engines:', torch.backends.quantized.supported_engines)
print('Selected quantization engine:', quantization_backend)

Supported quantization engines: ['onednn']
Selected quantization engine: onednn


In [8]:
preprocessing_check_input = torch.as_tensor(
    X_validation[:4],
    dtype=torch.float32,
)

with torch.no_grad():
    wrapper_logits = mobilenet_fp32(
        preprocessing_check_input
    )

    manual_inner_input = prepare_mobilenet_inner_inputs(
        mobilenet_fp32,
        X_validation[:4],
    )

    inner_logits = mobilenet_fp32.model(
        manual_inner_input
    )

preprocessing_max_abs_diff = torch.max(
    torch.abs(
        wrapper_logits - inner_logits
    )
).item()

print('Max absolute difference between wrapper preprocessing and manual inner preprocessing:')
print(preprocessing_max_abs_diff)

Max absolute difference between wrapper preprocessing and manual inner preprocessing:
0.0


In [9]:
static_int8_model = build_static_int8_model(
    mobilenet_fp32,
    X_validation,
    quantization_backend=quantization_backend,
    calibration_sample_count=STATIC_CALIBRATION_SAMPLE_COUNT,
    batch_size=MOBILENET_BATCH_SIZE,
)

dynamic_linear_model = build_dynamic_linear_model(
    mobilenet_fp32
)

classifier_dynamic_model = build_classifier_dynamic_model(
    mobilenet_fp32
)

example_deployment_input = torch.as_tensor(
    X_test[:1],
    dtype=torch.float32,
)

model_registry = {
    'mobilenet_v2_fp32': {
        'model': mobilenet_fp32,
        'path': FP32_MOBILENET_DEPLOYMENT_PATH,
        'quantization': 'none',
    },
    'mobilenet_v2_static_int8': {
        'model': static_int8_model,
        'path': STATIC_INT8_MOBILENET_MODEL_PATH,
        'quantization': 'static_int8_core',
    },
    'mobilenet_v2_dynamic_linear': {
        'model': dynamic_linear_model,
        'path': DYNAMIC_LINEAR_MOBILENET_MODEL_PATH,
        'quantization': 'dynamic_linear',
    },
    'mobilenet_v2_classifier_dynamic': {
        'model': classifier_dynamic_model,
        'path': CLASSIFIER_DYNAMIC_MOBILENET_MODEL_PATH,
        'quantization': 'classifier_dynamic',
    },
}

for model_name, model_info in model_registry.items():
    artifact_type = save_deployment_model(
        model_info['model'],
        model_info['path'],
        example_deployment_input,
    )

    model_info['artifact_type'] = artifact_type

print('Static calibration samples:', min(STATIC_CALIBRATION_SAMPLE_COUNT, len(X_validation)))
print('Saved deployment artifacts:')

for model_name, model_info in model_registry.items():
    print(model_name, '->', model_info['path'])

C:\Users\diots\.conda\envs\musicgenre\lib\site-packages\torch\ao\quantization\qconfig_mapping.py:74: UserWarning: Default qconfig of oneDNN backend with reduce_range of false may have accuracy issues on CPU without Vector Neural Network Instruction support.
  qconfig = get_default_qconfig(backend, version)
W0617 10:56:07.783000 32764 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\Softuni ML course\Project\music-genre-classification\src\models\mobilenet_model.py:100: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[1] == 1:


Static calibration samples: 1024
Saved deployment artifacts:
mobilenet_v2_fp32 -> C:\Softuni ML course\Project\music-genre-classification\models\quantized\mobilenet_v2_fp32_deployment.pt
mobilenet_v2_static_int8 -> C:\Softuni ML course\Project\music-genre-classification\models\quantized\mobilenet_v2_static_int8_quantized.pt
mobilenet_v2_dynamic_linear -> C:\Softuni ML course\Project\music-genre-classification\models\quantized\mobilenet_v2_dynamic_linear_quantized.pt
mobilenet_v2_classifier_dynamic -> C:\Softuni ML course\Project\music-genre-classification\models\quantized\mobilenet_v2_classifier_dynamic_quantized.pt


## Quantized Model Evaluation

Each quantized candidate is evaluated on the same cached test split used for the FP32 baseline.

The test split is not used for calibration or model selection. It is used only for final comparison.

In [10]:
evaluation_results = [
    baseline_result
]

for model_name in [
    'mobilenet_v2_static_int8',
    'mobilenet_v2_dynamic_linear',
    'mobilenet_v2_classifier_dynamic',
]:
    result = evaluate_torch_model(
        model_registry[model_name]['model'],
        X_test,
        y_test,
        class_names,
        model_name=model_name,
        split='test',
        batch_size=MOBILENET_BATCH_SIZE,
        device=EVALUATION_DEVICE,
    )

    evaluation_results.append(
        result
    )

metrics_comparison = pd.DataFrame([
    result['summary']
    for result in evaluation_results
])

baseline_metrics = metrics_comparison.loc[
    metrics_comparison['model'] == 'mobilenet_v2_fp32'
].iloc[0]

for metric in [
    'accuracy',
    'macro_f1',
    'weighted_f1',
]:
    metrics_comparison[f'{metric}_delta_vs_fp32'] = (
        metrics_comparison[metric]
        - baseline_metrics[metric]
    )

metrics_comparison

,model,split,accuracy,macro_f1,weighted_f1,accuracy_delta_vs_fp32,macro_f1_delta_vs_fp32,weighted_f1_delta_vs_fp32
0,mobilenet_v2_fp32,test,0.813296,0.435781,0.770928,0.000000,0.000000,0.000000
1,mobilenet_v2_static_int8,test,0.352900,0.156846,0.339366,-0.460396,-0.278935,-0.431562
2,mobilenet_v2_dynamic_linear,test,0.813296,0.438572,0.771542,0.000000,0.002791,0.000614
3,mobilenet_v2_classifier_dynamic,test,0.813296,0.438572,0.771542,0.000000,0.002791,0.000614


In [11]:
report_tables = []

for result in evaluation_results:
    report = pd.DataFrame(
        result['classification_report']
    ).transpose()

    report.insert(
        0,
        'model',
        result['summary']['model'],
    )

    report.insert(
        1,
        'split',
        result['summary']['split'],
    )

    report.insert(
        2,
        'label',
        report.index,
    )

    report_tables.append(
        report.reset_index(
            drop=True
        )
    )

classification_reports = pd.concat(
    report_tables,
    ignore_index=True,
)

MOBILENET_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

classification_reports.to_csv(
    QUANTIZATION_CLASSIFICATION_REPORTS_PATH,
    index=False,
)

print('Saved classification reports:', QUANTIZATION_CLASSIFICATION_REPORTS_PATH)
classification_reports.head()

Saved classification reports: C:\Softuni ML course\Project\music-genre-classification\outputs\mobilenet\mobilenet_quantization_classification_reports.csv


,model,split,label,precision,recall,f1-score,support
0,mobilenet_v2_fp32,test,blues,1.000000,0.153846,0.266667,26.0
1,mobilenet_v2_fp32,test,classical,0.796296,0.796296,0.796296,108.0
2,mobilenet_v2_fp32,test,country,0.000000,0.000000,0.000000,42.0
3,mobilenet_v2_fp32,test,hiphop,0.804348,0.766839,0.785146,193.0
4,mobilenet_v2_fp32,test,jazz,0.750000,0.166667,0.272727,72.0


## Model Size Comparison

All rows use deployment artifacts saved by this notebook.

This gives a consistent size comparison across FP32, static INT8, dynamic Linear-only, and classifier-only dynamic variants.

In [12]:
size_comparison = pd.DataFrame([
    {
        'model': model_name,
        'quantization': model_info['quantization'],
        'artifact_type': model_info['artifact_type'],
        'path': str(model_info['path']),
        'size_mb': size_in_mb(model_info['path']),
    }
    for model_name, model_info in model_registry.items()
])

fp32_size = size_comparison.loc[
    size_comparison['model'] == 'mobilenet_v2_fp32',
    'size_mb',
].iloc[0]

size_comparison['size_reduction_percent_vs_fp32'] = (
    (fp32_size - size_comparison['size_mb'])
    / fp32_size
    * 100
)

size_comparison.to_csv(
    QUANTIZATION_SIZE_COMPARISON_PATH,
    index=False,
)

print('Saved size comparison:', QUANTIZATION_SIZE_COMPARISON_PATH)
size_comparison

Saved size comparison: C:\Softuni ML course\Project\music-genre-classification\outputs\mobilenet\mobilenet_quantization_size_comparison.csv


,model,quantization,artifact_type,path,size_mb,size_reduction_percent_vs_fp32
0,mobilenet_v2_fp32,none,torchscript,C:\Softuni ML course\Project\music-genre-class...,8.489531,0.000000
1,mobilenet_v2_static_int8,static_int8_core,torchscript,C:\Softuni ML course\Project\music-genre-class...,2.504530,70.498605
2,mobilenet_v2_dynamic_linear,dynamic_linear,torchscript,C:\Softuni ML course\Project\music-genre-class...,8.465987,0.277322
3,mobilenet_v2_classifier_dynamic,classifier_dynamic,torchscript,C:\Softuni ML course\Project\music-genre-class...,8.466495,0.271346


## Inference Speed Comparison

Inference speed is measured on CPU for all variants using the same cached test subset.

A short warm-up is performed before timing. The table reports average latency over repeated full passes through the timing subset.

In [13]:
timing_sample_count = min(
    TIMING_SAMPLE_COUNT,
    len(X_test),
)

X_timing = X_test[:timing_sample_count]

speed_comparison = pd.DataFrame([
    {
        'model': model_name,
        **measure_inference_speed(
            model_info['model'],
            X_timing,
            MOBILENET_BATCH_SIZE,
            repeats=TIMING_REPEATS,
        ),
    }
    for model_name, model_info in model_registry.items()
])

fp32_ms = speed_comparison.loc[
    speed_comparison['model'] == 'mobilenet_v2_fp32',
    'milliseconds_per_sample',
].iloc[0]

speed_comparison['latency_reduction_percent_vs_fp32'] = (
    (fp32_ms - speed_comparison['milliseconds_per_sample'])
    / fp32_ms
    * 100
)

speed_comparison.to_csv(
    QUANTIZATION_SPEED_COMPARISON_PATH,
    index=False,
)

print('Saved speed comparison:', QUANTIZATION_SPEED_COMPARISON_PATH)
speed_comparison

Saved speed comparison: C:\Softuni ML course\Project\music-genre-classification\outputs\mobilenet\mobilenet_quantization_speed_comparison.csv


,model,samples,repeats,mean_seconds,std_seconds,samples_per_second,milliseconds_per_sample,latency_reduction_percent_vs_fp32
0,mobilenet_v2_fp32,256,5,4.958078,0.081982,51.632916,19.367490,0.000000
1,mobilenet_v2_static_int8,256,5,2.798556,0.198682,91.475734,10.931861,43.555615
2,mobilenet_v2_dynamic_linear,256,5,4.912018,0.061148,52.117069,19.187572,0.928973
3,mobilenet_v2_classifier_dynamic,256,5,4.906125,0.052088,52.179672,19.164551,1.047833


## Results Discussion

The final comparison combines predictive performance with deployment-oriented measurements.

The static INT8 model should be interpreted cautiously. If its macro F1 remains far below FP32, the quantization path is unsuccessful even if the artifact is smaller or faster.

The dynamic and classifier-only variants are safer because they keep the convolutional feature extractor in FP32. They may preserve macro F1 better, but they usually offer smaller deployment benefits because MobileNetV2 is dominated by convolutional layers.

In [14]:
metrics_comparison.to_csv(
    QUANTIZATION_METRICS_SUMMARY_PATH,
    index=False,
)

deployment_summary = (
    metrics_comparison
    .merge(
        size_comparison[[
            'model',
            'quantization',
            'artifact_type',
            'size_mb',
            'size_reduction_percent_vs_fp32',
        ]],
        on='model',
        how='left',
    )
    .merge(
        speed_comparison[[
            'model',
            'milliseconds_per_sample',
            'samples_per_second',
            'latency_reduction_percent_vs_fp32',
        ]],
        on='model',
        how='left',
    )
)

quantized_candidates = deployment_summary[
    deployment_summary['model'] != 'mobilenet_v2_fp32'
].copy()

quantized_candidates['macro_f1_drop'] = (
    baseline_metrics['macro_f1']
    - quantized_candidates['macro_f1']
)

quantized_candidates['preserves_macro_f1'] = (
    quantized_candidates['macro_f1_drop']
    <= ACCEPTABLE_MACRO_F1_DROP
)

quantized_candidates['has_deployment_benefit'] = (
    (quantized_candidates['size_reduction_percent_vs_fp32'] > 0)
    | (quantized_candidates['latency_reduction_percent_vs_fp32'] > 0)
)

successful_candidates = quantized_candidates[
    quantized_candidates['preserves_macro_f1']
    & quantized_candidates['has_deployment_benefit']
]

h7_supported = not successful_candidates.empty

if h7_supported:
    best_candidate = successful_candidates.sort_values(
        ['macro_f1', 'latency_reduction_percent_vs_fp32'],
        ascending=[False, False],
    ).iloc[0]

    print('H7 supported:', True)
    print('Best quantized candidate:', best_candidate['model'])
    print('Macro F1 drop:', best_candidate['macro_f1_drop'])
else:
    best_candidate = quantized_candidates.sort_values(
        'macro_f1',
        ascending=False,
    ).iloc[0]

    print('H7 supported:', False)
    print('No quantized candidate preserved macro F1 within the configured tolerance while improving deployment cost.')
    print('Best quantized candidate by macro F1:', best_candidate['model'])
    print('Macro F1 drop:', best_candidate['macro_f1_drop'])

print('Saved quantization metrics summary:', QUANTIZATION_METRICS_SUMMARY_PATH)
deployment_summary

H7 supported: True
Best quantized candidate: mobilenet_v2_classifier_dynamic
Macro F1 drop: -0.0027910297037503917
Saved quantization metrics summary: C:\Softuni ML course\Project\music-genre-classification\outputs\mobilenet\mobilenet_quantization_metrics_summary.csv


,model,split,accuracy,macro_f1,weighted_f1,accuracy_delta_vs_fp32,macro_f1_delta_vs_fp32,weighted_f1_delta_vs_fp32,quantization,artifact_type,size_mb,size_reduction_percent_vs_fp32,milliseconds_per_sample,samples_per_second,latency_reduction_percent_vs_fp32
0,mobilenet_v2_fp32,test,0.813296,0.435781,0.770928,0.000000,0.000000,0.000000,none,torchscript,8.489531,0.000000,19.367490,51.632916,0.000000
1,mobilenet_v2_static_int8,test,0.352900,0.156846,0.339366,-0.460396,-0.278935,-0.431562,static_int8_core,torchscript,2.504530,70.498605,10.931861,91.475734,43.555615
2,mobilenet_v2_dynamic_linear,test,0.813296,0.438572,0.771542,0.000000,0.002791,0.000614,dynamic_linear,torchscript,8.465987,0.277322,19.187572,52.117069,0.928973
3,mobilenet_v2_classifier_dynamic,test,0.813296,0.438572,0.771542,0.000000,0.002791,0.000614,classifier_dynamic,torchscript,8.466495,0.271346,19.164551,52.179672,1.047833


## Conclusion

This notebook optimized the saved MobileNetV2 model for deployment without retraining and without regenerating any preprocessing artifacts.

The completed workflow includes:

- loading cached validation and test Mel Spectrogram tensors,
- loading the saved MobileNetV2 label encoder and FP32 weights,
- evaluating the original FP32 model,
- verifying that manual MobileNetV2-core preprocessing matches the FP32 wrapper preprocessing,
- applying full static INT8 quantization with validation-only calibration,
- applying dynamic Linear-only quantization,
- applying classifier-only dynamic quantization,
- saving deployment artifacts for all compared variants,
- comparing model size, CPU inference speed, accuracy, macro F1, and weighted F1.

Negative results are not hidden. If full static INT8 quantization reduces model size or latency but severely lowers macro F1, it should be reported as an unsuccessful deployment optimization.

H7 is supported only if at least one quantized candidate provides a deployment benefit while keeping macro F1 within the configured tolerance of the FP32 MobileNetV2 baseline.

The full static INT8 MobileNetV2 achieved the largest reduction in model size and inference latency, but caused a severe macro F1 degradation and is therefore unsuitable for deployment in this project.

Dynamic quantization and classifier-only dynamic quantization preserved predictive performance while providing smaller deployment benefits.

The dynamic Linear-only variant emerged as the most practical deployment candidate because it preserved macro F1 within the predefined tolerance while improving CPU inference speed.